In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════════════════╗
# ║ lemmalog — MCC evaluation of LLM-free extraction (gline-rs + NegEx + glinker-rs) (Colab)  ║
# ║ Steps: deps → clone → Lean4 proofs → cargo tests → GLiNER multitask ONNX → linker export  ║
# ║        → entity DB + pipeline config → extraction_eval → MCC (cross-checked with sklearn)  ║
# ╚══════════════════════════════════════════════════════════════════════════════════════════╝
LEMMALOG_REPO   = "https://github.com/dhaneshkk/lemmalog"
LEMMALOG_BRANCH = "main"
GLINKER_REPO    = "https://github.com/dhaneshkk/glinker-rs"
GLINER_MODEL    = "onnx-community/gliner-multitask-large-v0.5"   # token-mode multitask (NER + relations)
# glinker-rs L3 runs span-mode GLiNER only; the first candidate whose config is not
# token_level is exported. knowledgator/gliner-linker-base-v1.0 is token-level in
# gliner 0.2.30 (no span_idx), so it is skipped.
LINKER_CANDIDATES = ["knowledgator/gliner-linker-base-v1.0", "knowledgator/gliner-linker-large-v1.0",
                     "knowledgator/modern-gliner-bi-base-v1.0", "knowledgator/gliner-bi-base-v1.0",
                     "urchade/gliner_medium-v2.1"]
L3_THRESHOLD    = 0.05   # glinker-rs drops predictions below this before lemmalog sees them
SWEEP           = "0.1,0.2,0.3,0.4,0.5"   # GlinkerLinker.min_confidence values to score
WORK            = "/content/lemmalog_eval"

import os, sys, json, subprocess, shutil, pathlib, glob, re
def sh(cmd, check=True, env=None, quiet=False):
    print(f"$ {cmd[:160]}")
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, env={**os.environ, **(env or {})})
    out = (r.stdout + r.stderr).strip()
    if out and (not quiet or r.returncode): print(out[-6000:])
    if check and r.returncode: raise RuntimeError(f"failed: {cmd}")
    return r.stdout

# ── 0. dependencies ───────────────────────────────────────────────────────────────────────
sh(f"{sys.executable} -m pip install -q gliner==0.2.30 onnx onnxruntime==1.20.1 huggingface_hub scikit-learn tokenizers", quiet=True)
if not os.path.exists(os.path.expanduser("~/.cargo/bin/cargo")):
    sh("curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal", quiet=True)
if not os.path.exists(os.path.expanduser("~/.elan/bin/lean")):
    sh("curl -sSfL https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y --default-toolchain leanprover/lean4:v4.15.0", quiet=True)
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.path.expanduser("~/.elan/bin") + ":" + os.environ["PATH"]
import onnxruntime as _ort
ORT_DIR = os.path.join(os.path.dirname(_ort.__file__), "capi")
ORT_SO = sorted(glob.glob(os.path.join(ORT_DIR, "libonnxruntime.so*")))[-1]
ENV = {"ORT_LIB_LOCATION": ORT_DIR, "ORT_DYLIB_PATH": ORT_SO, "RUST_BACKTRACE": "0"}
os.environ.update(ENV)
FEATURES = "gliner-load-dynamic,glinker"

# ── 1. sources ────────────────────────────────────────────────────────────────────────────
os.makedirs(WORK, exist_ok=True)
LM, GK = f"{WORK}/lemmalog", f"{WORK}/glinker-rs"
if not os.path.exists(LM): sh(f"git clone -q --depth 1 -b {LEMMALOG_BRANCH} {LEMMALOG_REPO} {LM}")
if not os.path.exists(GK): sh(f"git clone -q --depth 1 {GLINKER_REPO} {GK}")

# ── 2. Lean4 formal guarantees (prompt offsets, span safety, direction, resolver, scoring) ─
sh(f"cd {LM}/lean && lean Extraction.lean")  # fast; re-checked every run
print("Lean4: all theorems in lean/Extraction.lean type-check ✔")

# ── 3. cargo tests: spec obligations + review regressions + glinker-rs e2e on a real session
sh(f"cd {LM} && cargo test --release --features {FEATURES} --lib --test extraction_test --test extraction_review_test --test extraction_followup_test 2>&1 | grep -E 'test result|FAILED|panicked'", env=ENV)
SYN = f"{WORK}/synthetic"
if not os.path.exists(f"{SYN}/models/linker/model.onnx"):
    sh(f"{sys.executable} {GK}/tools/make_synthetic.py {SYN}/models/linker")
    for f in ("simple.json", "entities.jsonl"): shutil.copy(f"{GK}/tests/fixtures/{f}", SYN)
sh(f"cd {SYN} && GLINKER_E2E_CONFIG={SYN}/simple.json cargo test --release --manifest-path {LM}/Cargo.toml --features {FEATURES} --test glinker_e2e_test 2>&1 | grep -E 'test result|FAILED|panicked|skipped'", env=ENV)

# ── 4. GLiNER multitask (token mode) ONNX for gline-rs ─────────────────────────────────────
from huggingface_hub import hf_hub_download, list_repo_files
_files = list_repo_files(GLINER_MODEL)
_onnx = "onnx/model.onnx" if "onnx/model.onnx" in _files else sorted(f for f in _files if f.endswith(".onnx") and "quant" not in f and "fp16" not in f)[0]
_tok = "tokenizer.json" if "tokenizer.json" in _files else sorted(f for f in _files if f.endswith("tokenizer.json"))[0]
print(f"{GLINER_MODEL}: using {_tok}, {_onnx}")
GL_TOK = hf_hub_download(GLINER_MODEL, _tok)
GL_ONNX = hf_hub_download(GLINER_MODEL, _onnx)

# ── 5. export the glinker-rs L3 linker (exporter from glinker-rs/notebooks) ────────────────
import torch
from gliner import GLiNER

def load_gliner(model_id):
    m = GLiNER.from_pretrained(model_id).eval(); m.to("cpu")
    lt = getattr(m.data_processor, "labels_tokenizer", None)
    if lt is not None and lt.model_max_length > 100000:   # same fix GLinker's L3Component applies
        lt.model_max_length = 512
    return m

def export_gliner(model_id, out_dir):
    out = pathlib.Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    m = load_gliner(model_id)
    if (out / "model.onnx").exists(): return m
    core = m.model.eval()
    probe = ["person", "organization", "country"]
    batch = m._build_dummy_batch(labels=probe)
    base = ["input_ids", "attention_mask", "words_mask", "text_lengths", "span_idx", "span_mask"]
    axes = {"input_ids": {0: "batch", 1: "seq"}, "attention_mask": {0: "batch", 1: "seq"}, "words_mask": {0: "batch", 1: "seq"},
            "text_lengths": {0: "batch"}, "span_idx": {0: "batch", 1: "num_spans"}, "span_mask": {0: "batch", 1: "num_spans"},
            "logits": {0: "batch", 1: "words", 2: "width", 3: "classes"}}
    bi = getattr(m.config, "labels_encoder", None) is not None
    if bi:
        emb = m.encode_labels(probe).cpu()
        class TextGraph(torch.nn.Module):
            def __init__(s, c): super().__init__(); s.c = c
            def forward(s, input_ids, attention_mask, words_mask, text_lengths, span_idx, span_mask, labels_embeddings):
                o = s.c(input_ids=input_ids, attention_mask=attention_mask, words_mask=words_mask, text_lengths=text_lengths,
                        span_idx=span_idx, span_mask=span_mask, labels_embeds=labels_embeddings)
                return o.logits if hasattr(o, "logits") else o[0]
        torch.onnx.export(TextGraph(core), tuple(batch[k] for k in base) + (emb,), str(out / "model.onnx"),
                          input_names=base + ["labels_embeddings"], output_names=["logits"],
                          dynamic_axes={**axes, "labels_embeddings": {0: "classes"}}, opset_version=19, dynamo=False)
        class LabelGraph(torch.nn.Module):
            def __init__(s, c): super().__init__(); s.c = c
            def forward(s, input_ids, attention_mask):
                return s.c.token_rep_layer.encode_labels(input_ids, attention_mask)
        lt = m.data_processor.labels_tokenizer(probe, return_tensors="pt", padding=True)
        torch.onnx.export(LabelGraph(core), (lt["input_ids"], lt["attention_mask"]), str(out / "labels_encoder.onnx"),
                          input_names=["input_ids", "attention_mask"], output_names=["embeddings"],
                          dynamic_axes={"input_ids": {0: "n", 1: "t"}, "attention_mask": {0: "n", 1: "t"}, "embeddings": {0: "n"}},
                          opset_version=19, dynamo=False)
        tmp = out / "_labels_tok"; m.data_processor.labels_tokenizer.save_pretrained(tmp)
        shutil.copy(tmp / "tokenizer.json", out / "labels_tokenizer.json")
    else:
        class UniGraph(torch.nn.Module):
            def __init__(s, c): super().__init__(); s.c = c
            def forward(s, input_ids, attention_mask, words_mask, text_lengths, span_idx, span_mask):
                o = s.c(input_ids=input_ids, attention_mask=attention_mask, words_mask=words_mask, text_lengths=text_lengths,
                        span_idx=span_idx, span_mask=span_mask)
                return o.logits if hasattr(o, "logits") else o[0]
        torch.onnx.export(UniGraph(core), tuple(batch[k] for k in base), str(out / "model.onnx"),
                          input_names=base, output_names=["logits"], dynamic_axes=axes, opset_version=19, dynamo=False)
    m.config.to_json_file(str(out / "gliner_config.json"))
    tmp = out / "_tok"; m.data_processor.transformer_tokenizer.save_pretrained(tmp)
    shutil.copy(tmp / "tokenizer.json", out / "tokenizer.json")
    print(f"exported {model_id} → {out} ({'bi' if bi else 'uni'}-encoder)")
    return m

MODELS = os.path.join(WORK, "models")
LINKER_MODEL = None
for cand in LINKER_CANDIDATES:
    try:
        cfg = json.load(open(hf_hub_download(cand, "gliner_config.json")))
    except Exception as e:
        print(f"linker {cand}: no gliner_config.json ({type(e).__name__}), skipped"); continue
    mode = cfg.get("span_mode"); bi = cfg.get("labels_encoder") is not None
    print(f"linker {cand}: span_mode={mode} {'bi' if bi else 'uni'}-encoder")
    if mode == "token_level":
        print("  token-level: glinker-rs cannot run it, skipped"); continue
    try:
        shutil.rmtree(f"{MODELS}/linker", ignore_errors=True)
        export_gliner(cand, f"{MODELS}/linker"); LINKER_MODEL = cand; break
    except Exception as e:
        print(f"  export failed: {type(e).__name__}: {e}")
if LINKER_MODEL is None:
    print("⚠ no linker exported: only the GLiNER + NegEx run (gold alias table) will be scored")

# ── 6. entity DBs from the gold entity table + one glinker-rs pipeline config each ──────────
#      "type": description = GLiNER label only ("person"); "rich": the gold set's description
#      column ("human; a person named Bob Smith"). Same aliases, so the exact-alias table is shared.
GOLD = f"{LM}/benchmarks/extraction_gold.tsv"
PIPES = {}
if LINKER_MODEL:
    rows = [l.rstrip("\n").split("\t") for l in open(GOLD) if l.startswith("@\t")]
    for variant in ("type", "rich"):
        ents = []
        for f in rows:
            aliases = f[3].split("|")
            desc = f[4] if variant == "rich" and len(f) > 4 and f[4] else f[2]
            ents.append({"entity_id": f[1], "label": aliases[0], "aliases": aliases[1:], "description": desc,
                         "entity_type": f[2], "popularity": 50})
        open(f"{WORK}/entities_{variant}.jsonl", "w").write("\n".join(json.dumps(e) for e in ents))
        PIPES[variant] = f"{WORK}/pipeline_{variant}.json"
        json.dump({"name": f"lemmalog-eval-{variant}", "template": "{label}: {description}",
                   "entities": f"{WORK}/entities_{variant}.jsonl", "precompute_embeddings": True, "l1": {"type": "none"},
                   "l3": {"model_dir": f"{MODELS}/linker", "model_id": LINKER_MODEL, "threshold": L3_THRESHOLD}},
                  open(PIPES[variant], "w"), indent=1)

# ── 7. run the evaluation ──────────────────────────────────────────────────────────────────
sh(f"cd {LM} && cargo build --release --features {FEATURES} --example extraction_eval", env=ENV, quiet=True)
ARGS = " ".join(f"--pipeline glinker-{v}={p}" for v, p in PIPES.items())
if PIPES: ARGS += f" --entities {WORK}/entities_rich.jsonl --sweep {SWEEP}"
ARGS += " --spans"
out = sh(f"cd {WORK} && {LM}/target/release/examples/extraction_eval {GL_TOK} {GL_ONNX} {GOLD} {ARGS}", env=ENV)

# ── 8. MCC, cross-checked with sklearn ─────────────────────────────────────────────────────
from sklearn.metrics import matthews_corrcoef
rows = []
for name, tp, tn, fp, fn, mcc in re.findall(r"== (.+)\nTP=(\d+) TN=(\d+) FP=(\d+) FN=(\d+).*MCC=(-?[\d.]+)", out):
    tp, tn, fp, fn = map(int, (tp, tn, fp, fn))
    y_true = [1] * tp + [0] * tn + [0] * fp + [1] * fn
    y_pred = [1] * tp + [0] * tn + [1] * fp + [0] * fn
    sk = matthews_corrcoef(y_true, y_pred)
    assert abs(sk - float(mcc)) < 1e-3, (name, sk, mcc)
    rows.append((name, tp, tn, fp, fn, float(mcc)))
print("\n{:<45} {:>4} {:>4} {:>4} {:>4} {:>7}".format("run", "TP", "TN", "FP", "FN", "MCC"))
for r in rows: print("{:<45} {:>4} {:>4} {:>4} {:>4} {:>7.3f}".format(*r))
print("(the gold set has 40 labelled facts: one fact moves MCC by ~0.05; a threshold picked here is tuned on the test set)")
print("MCC matches sklearn.metrics.matthews_corrcoef ✔")
